# SHAP-IMV — results across the three examples

Everything the three `src/empirical/shap_imv/*.ipynb` notebooks produced, gathered in one place: the top-5 feature-selection comparison as one LaTeX table, the SHAP-IMV ranking of the variables in each example as a second one, and a 3×3 figure showing that ranking.

Nothing is recomputed here. The tables are read back from `output/examples/shap/` and the artefact cache, so this notebook is cheap to re-run and cannot disagree with the examples it summarises. Run the three example notebooks first if a file is missing.

Both `.tex` files are written to `output/tables/` as complete `table` environments, ready for `\input`. They are regenerated on every run, so any edit belongs in the constants below rather than in the file — the citation keys in particular.

The figure style — panel size, palette, weights, sizes, edge colour — comes from `src/shared_style.py`, so any other notebook can import it and produce panels that sit beside these without retouching.

In [ ]:
import os, sys
from pathlib import Path

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "requirements.txt").is_file() and (path / "src").is_dir()),
    Path.cwd(),
)

def relative_path(path, *, start=None):
    """Display-only path, anchored so no absolute home path reaches an output."""
    anchor = Path(start) if start is not None else PROJECT_ROOT
    try:
        relative = os.path.relpath(Path(path).expanduser().resolve(),
                                   start=Path(anchor).expanduser().resolve())
    except ValueError:  # different Windows drives cannot be relativized
        relative = Path(path).name
    return Path(relative).as_posix()

# The style module lives in src/ so every notebook, wherever it sits, imports
# the same one.
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import shared_style
from figure_utils import save_publication_figure as save_figure

CACHE = Path(os.environ.get("IMV_CACHE_HOME", Path.home() / ".cache" / "imv"))
ARTIFACTS = Path(os.environ.get("IMV_ARTIFACT_CACHE", CACHE / "notebook_artifacts"))
OUTPUT = PROJECT_ROOT / "output" / "examples" / "shap"
OUTPUT.mkdir(parents=True, exist_ok=True)
TABLES = PROJECT_ROOT / "output" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES = Path(os.environ.get("IMV_FIGURE_DIR", PROJECT_ROOT / "output" / "figures"))
FIGURES.mkdir(parents=True, exist_ok=True)

# Datasets in the column order of the table; each notebook is named for its
# example and its artefacts for the dataset slug inside it.
EXAMPLES = {"shap_imv_titanic": ("titanic", "Titanic"),
            "shap_imv_breast_cancer": ("breast_cancer", "Breast Cancer"),
            "shap_imv_adult_income": ("adult_income", "Adult Income")}
MODEL_NAMES = ("logistic_regression", "xgboost", "lightgbm")
MODEL_LABELS = {"logistic_regression": "LR", "xgboost": "XGB", "lightgbm": "LGBM"}
MODEL_TITLES = {"logistic_regression": "Logistic Regression",
                "xgboost": "XGBoost", "lightgbm": "LightGBM"}
METHOD_NAMES = ("shap_imv", "shap", "lime", "anchors")
METHOD_LABELS = {"shap_imv": "SHAP-IMV", "shap": "SHAP", "lime": "LIME",
                 "anchors": "Anchors"}
# The manuscript cites the three baselines in the text rather than in the table,
# so no key is emitted by default. Fill this in — {"shap": r"\cite{SHAP}"} and so
# on — to put them back; the keys have to exist in the bibliography, so edit them
# here rather than in the generated .tex, which every run overwrites.
METHOD_CITATIONS = {}
METRICS = {"accuracy_mean": "Accuracy", "precision_mean": "Precision"}
TOP_K = 5

# Short display names for the ranking table; anything missing falls through to
# the raw column name, escaped.
FEATURE_LABELS = {"AgeClass": r"Age$\times$Class", "sex_female": "Sex",
                  "age": "Age", "education-num": "Education",
                  "hours-per-week": "Hours/week",
                  "capital-gain": "Capital gain", "married": "Married",
                  "radius1": "Radius", "texture1": "Texture",
                  "smoothness1": "Smoothness", "compactness1": "Compactness",
                  "symmetry1": "Symmetry", "fractal_dimension1": "Fractal dim."}

def installed_fonts(families):
    """The families of the shared stack this machine actually has.

    Matplotlib falls back through `font.family` silently only when every name
    resolves; a missing one is logged as `findfont: Font family 'X' not found`
    once per text style, which buries a notebook's real output. Dropping the
    absent names first leaves the same rendering without the noise, and the
    Matplotlib-bundled DejaVu Sans is always there to catch an empty stack.
    """
    available = {font.name for font in font_manager.fontManager.ttflist}
    return [family for family in families if family in available] or ["DejaVu Sans"]

FONT_STACK = installed_fonts(shared_style.FONT_FAMILY)
shared_style.apply(**{"font.family": FONT_STACK, "font.sans-serif": FONT_STACK})
print(f"fonts: {', '.join(FONT_STACK)}")
print(f"style: {shared_style.PALETTE} palette, "
      f"{shared_style.PANEL_WIDTH}×{shared_style.PANEL_HEIGHT}in panels, "
      f"base font {shared_style.BASE_FONT_SIZE}pt")
print(f"reading from {relative_path(OUTPUT)}/ and {relative_path(ARTIFACTS)}/")

## 1. Load what the examples wrote

Each file is looked up under `output/examples/shap/` first and in the artefact cache second, so a checkout that ships only `output/` still renders everything: the per-feature SHAP-IMV summary lives in the cache, and where it is absent the ranking is recovered from the published top-5 table instead.

The per-example tables are then concatenated into two cross-example CSVs, which is the one thing this notebook adds rather than repeats.

In [ ]:
def write_csv_atomic(table, path, *, index=False):
    """Replace a CSV only after its complete contents reach the same directory."""
    temporary = path.with_name(f".{path.name}.{os.getpid()}.tmp")
    try:
        table.to_csv(temporary, index=index)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

def read_result(example, filename, **read_kwargs):
    """Read one artefact, preferring the published copy under output/."""
    candidates = [OUTPUT / filename, ARTIFACTS / example / "results" / filename]
    for path in candidates:
        if path.is_file():
            return pd.read_csv(path, **read_kwargs)
    raise FileNotFoundError(
        f"{filename} is missing; run src/empirical/shap_imv/{example}.ipynb "
        f"first (looked in {', '.join(relative_path(path) for path in candidates)})")

def ranking_from_top_k(top_k):
    """The SHAP-IMV ranking recovered from the published top-k table.

    The table keeps the selected features in order with their values and the
    dropped ones without, so the ranking survives in full and only the values
    below the cut are lost.
    """
    rows = []
    for _, row in top_k[top_k["method"] == "shap_imv"].iterrows():
        selected = row["selected_features"].split(";")
        values = [float(value) for value in row["importance"].split(";")]
        dropped = [name for name in str(row["dropped_features"]).split(";") if name]
        rows += [{"model": row["model"], "feature": name, "mean": value,
                  "std": np.nan} for name, value in zip(selected, values)]
        rows += [{"model": row["model"], "feature": name, "mean": np.nan,
                  "std": np.nan} for name in dropped]
    return pd.DataFrame(rows)

def load_ranking(example, dataset, top_k):
    """Mean SHAP-IMV per feature and estimator, ranked, for one example."""
    try:
        summary = read_result(example, f"{dataset}_shap_imv_summary.csv")
    except FileNotFoundError:
        summary = ranking_from_top_k(top_k)
        print(f"  {example}: no per-feature summary in the cache; ranking read "
              f"off the top-{TOP_K} table, so the dropped features carry no value")
    summary = summary[summary["model"].isin(MODEL_NAMES)].copy()
    # Rank within an estimator, strongest first. A feature with no value — the
    # fallback above — sorts last, which is where the top-k table left it.
    summary["rank"] = (summary.groupby("model")["mean"]
                       .rank(ascending=False, method="first", na_option="bottom")
                       .astype(int))
    return summary.sort_values(["model", "rank"]).reset_index(drop=True)

def load_example(example, dataset, title):
    """One example's top-k selection table and its SHAP-IMV feature ranking."""
    top_k = read_result(example, f"{example}_top_{TOP_K}_features.csv")
    missing = sorted({(model, method) for model in MODEL_NAMES
                      for method in METHOD_NAMES}
                     - set(map(tuple, top_k[["model", "method"]].to_numpy())))
    if missing:
        raise ValueError(f"{example}: no top-{TOP_K} results for {missing}")
    return {"example": example, "dataset": dataset, "title": title,
            "top_k": top_k, "ranking": load_ranking(example, dataset, top_k),
            "n_seeds": int(top_k["n_seeds"].iloc[0]),
            "n_splits": int(top_k["n_splits"].iloc[0])}

results = {example: load_example(example, dataset, title)
           for example, (dataset, title) in EXAMPLES.items()}
# Every number in both tables is a mean over the same protocol; a mismatch would
# make the captions below wrong rather than merely imprecise.
protocol = {(result["n_seeds"], result["n_splits"]) for result in results.values()}
if len(protocol) != 1:
    raise ValueError(f"the examples disagree on seeds × folds: {sorted(protocol)}")
N_SEEDS, N_SPLITS = protocol.pop()
for result in results.values():
    features = result["ranking"].query("model == 'logistic_regression'")["feature"]
    print(f"{result['title']:<15} {len(features)} features, {N_SEEDS} seeds × "
          f"{N_SPLITS} folds: {', '.join(features)}")

combined_top_k = pd.concat([result["top_k"] for result in results.values()],
                           ignore_index=True)
combined_ranking = pd.concat(
    [result["ranking"].assign(example=result["example"], dataset=result["title"])
     for result in results.values()],
    ignore_index=True)[["example", "dataset", "model", "rank", "feature", "mean",
                        "std"]].rename(columns={"mean": "shap_imv_mean",
                                                "std": "shap_imv_std"})

for table, name in ((combined_top_k, "top_k_metrics"),
                    (combined_ranking, "feature_ranking")):
    write_csv_atomic(table, OUTPUT / f"shap_imv_results__{name}.csv")
print(f"wrote two cross-example tables to {relative_path(OUTPUT)}/")
combined_ranking.head(12)

## 2. Table 1 — top-5 feature selection

Each attribution method ranks the features; the top five of each ranking refit the same estimator, and accuracy and precision are averaged over ten folds for each of ten seeds. Columns are the three datasets and their unweighted average, rows the three estimators × four methods. The environment is the manuscript's: a full-width `table*` inside a `\resizebox`, with the second and fourth method of each block shaded as row striping.

**Bold** is the best value and <u>underline</u> the runner-up, within one estimator block and one column. Six features choosing five leaves the four methods room to agree, and they often do: where all four pick the same subset the four numbers are identical, and that column is left unmarked rather than declaring a winner among equals.

In [ ]:
# Row striping, applied to every cell of every second row of a block, as in the
# manuscript. It needs \usepackage[table]{xcolor}; set it to None to drop that
# dependency and leave the rows unshaded.
ROW_SHADE_COLOR = "gray!15"

def escape_latex(text):
    """Escape the characters that actually occur in these names."""
    replacements = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$",
                    "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}",
                    "~": r"\textasciitilde{}", "^": r"\textasciicircum{}"}
    return "".join(replacements.get(character, character) for character in str(text))

def method_label(method):
    """The method as the table prints it, with its citation where it has one."""
    citation = METHOD_CITATIONS.get(method, "")
    return METHOD_LABELS[method] + (f"~{citation}" if citation else "")

def specification_note():
    """The sentence both captions carry about the fixed feature specification.

    Exact Shapley attribution evaluates one model per subset, so each example
    fixes a small specification rather than taking the dataset whole. Saying so
    in the caption keeps a reader from reading the ranking as a complete one.
    """
    widths = {result["title"]: int(result["ranking"]["rank"].max())
              for result in results.values()}
    distinct = sorted(set(widths.values()))
    specification = (f"a fixed {distinct[0]}-variable specification"
                     if len(distinct) == 1 else
                     "a fixed specification ("
                     + ", ".join(f"{width} for {title}"
                                 for title, width in widths.items()) + ")")
    return (f"The variables of each dataset are {specification} carried over "
            f"from the example notebooks, not the dataset's full variable list: "
            f"exact Shapley attribution costs one model evaluation per subset "
            f"of them.")

def metrics_matrix():
    """Percentages per (estimator, method) × (dataset, metric), plus the average.

    The average is over datasets, not over the pooled folds: each dataset counts
    once whatever its size, which is how the column is read.
    """
    titles = [title for _, title in EXAMPLES.values()]
    wide = (combined_top_k
            .assign(dataset=combined_top_k["example"].map(lambda name: EXAMPLES[name][1]))
            .pivot_table(index=["model", "method"], columns="dataset",
                         values=list(METRICS)))
    matrix = pd.DataFrame({(title, metric): 100 * wide[(metric, title)]
                           for title in titles for metric in METRICS})
    for metric in METRICS:
        matrix[("Average", metric)] = np.mean(
            [matrix[(title, metric)] for title in titles], axis=0)
    matrix.columns = pd.MultiIndex.from_tuples(
        [(title, METRICS[metric]) for title, metric in matrix.columns],
        names=["dataset", "metric"])
    return matrix.reindex(pd.MultiIndex.from_product([MODEL_NAMES, METHOD_NAMES],
                                                     names=["model", "method"]))

def marked_cells(column):
    """`{method: latex}` for one column of one estimator block.

    Marks are applied to the printed value, so two methods that agree to two
    decimals are marked alike. A column whose four methods all agree separates
    nothing and is left plain.
    """
    values = column.round(2)
    distinct = sorted(set(values), reverse=True)
    best = distinct[0] if len(distinct) > 1 else None
    runner_up = distinct[1] if len(distinct) > 1 else None
    marks = {best: r"\textbf", runner_up: r"\underline"}
    return {method: (rf"{marks[value]}{{{value:.2f}}}" if value in marks
                     else f"{value:.2f}")
            for method, value in values.items()}

def shaded(cells, position):
    """Every second row of a block carries the stripe on each of its cells."""
    if position % 2 == 0 or not ROW_SHADE_COLOR:
        return cells
    return [rf"\cellcolor{{{ROW_SHADE_COLOR}}}{cell}" for cell in cells]

def metrics_latex(matrix):
    """The top-k selection table as the manuscript's full-width table environment."""
    datasets = list(dict.fromkeys(title for title, _ in matrix.columns))
    width = len(METRICS)
    lines = [
        r"\begin{table*}[h]",
        r"\centering",
        r"\tabcolsep=3pt",
        rf"\caption{{Results of the performance of SHAP-IMV, SHAP, LIME, and "
        rf"Anchors under three machine learning methods. LR represents Logistic "
        rf"Regression, XGB represents XGBoost, and LGBM represents LightGBM. For "
        rf"each dataset, the top {TOP_K} most important features as selected by "
        rf"the different baseline algorithms under each machine learning method "
        rf"are used to train the model. The accuracy and precision are averaged "
        rf"over {N_SPLITS} folds for each of {N_SEEDS} seeds, and the average "
        rf"weights the {len(EXAMPLES)} datasets equally. The best and second best "
        rf"are noted in bold font and underlined; a column on which all four "
        rf"methods agree is left unmarked. {specification_note()}}}",
        r"\resizebox{\textwidth}{!}{",
        r"    \begin{tabular}{" + "c" * (2 + len(matrix.columns)) + "}",
        r"        \toprule",
        r"         \multicolumn{2}{c}{\multirow{2}{*}{Methods}} & "
        + " & ".join(rf"\multicolumn{{{width}}}{{c}}{{{title}}}" for title in datasets)
        + r"\\",
        r"         &  & " + " & ".join(metric for _, metric in matrix.columns)
        + r" \\",
        r"        \midrule",
    ]
    for position, model in enumerate(MODEL_NAMES):
        if position:
            lines.append(r"        \midrule")
        block = matrix.loc[model]
        marks = {column: marked_cells(block[column]) for column in block.columns}
        lines.append(rf"        \multirow{{{len(METHOD_NAMES)}}}{{*}}{{{MODEL_LABELS[model]}}}")
        for offset, method in enumerate(METHOD_NAMES):
            cells = shaded([method_label(method)]
                           + [marks[column][method] for column in block.columns],
                           offset)
            lines.append("        & " + " & ".join(cells) + r"\\")
    lines += [
        r"        \bottomrule",
        rf"        \label{{{METRICS_LABEL}}}",
        r"    \end{tabular}}",
        r"    \vspace{-0.4cm}",
        r"\end{table*}",
    ]
    return "\n".join(lines)

def write_latex(body, path, *, packages):
    """Write a generated table, stamped with its source and its requirements."""
    header = (f"% Generated by {SOURCE}; do not edit by hand.\n"
              f"% Requires: {', '.join(packages)}.\n")
    temporary = path.with_name(f".{path.name}.{os.getpid()}.tmp")
    try:
        temporary.write_text(header + body + "\n")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)
    print(relative_path(path))
    return path

SOURCE = "src/plotter/shap_imv_results.ipynb"
# \resizebox is graphicx; the stripe is the table option of xcolor.
PACKAGES = [r"\usepackage{booktabs}", r"\usepackage{multirow}",
            r"\usepackage{graphicx}"] + ([r"\usepackage[table]{xcolor}"]
                                         if ROW_SHADE_COLOR else [])
METRICS_LABEL = "SHAPIMVTabulartable"
RANKING_LABEL = "SHAPIMVRankingtable"

metrics = metrics_matrix()
write_latex(metrics_latex(metrics), TABLES / "shap_imv_top_k.tex",
            packages=PACKAGES)
print(metrics_latex(metrics))
display(metrics.round(2)
        .rename(index=MODEL_LABELS, level="model")
        .rename(index=METHOD_LABELS, level="method"))

## 3. Table 2 — the SHAP-IMV ranking of the variables

One row per estimator, one column per rank: the features of each example ordered by their mean exact Shapley value with IMV as the coalition value, the mean itself in parentheses. This is the ranking whose top five feed the `SHAP-IMV` rows of Table 1, so the last column is the feature that method drops.

The ranking is the estimator's, not the dataset's: `Title` and `Sex` are near-redundant on Titanic, and Shapley symmetry splits the credit between them differently for a linear model than for a boosted one, which is why the top two swap between the rows.

In [ ]:
def feature_label(name):
    """The feature as the table prints it: a short label, or the escaped name."""
    return FEATURE_LABELS.get(name, escape_latex(name))

def ranking_cell(row):
    """One ranked feature: its label, and its mean SHAP-IMV where one survives."""
    label = feature_label(row["feature"])
    return label if pd.isna(row["mean"]) else f"{label} ({row['mean']:.3f})"

def ranking_latex():
    """The SHAP-IMV ranking of every example, in the same table environment."""
    depth = int(combined_ranking["rank"].max())
    lines = [
        r"\begin{table*}[h]",
        r"\centering",
        r"\tabcolsep=3pt",
        rf"\caption{{Variables of each dataset ranked by SHAP-IMV under three "
        rf"machine learning methods, with the mean exact Shapley value in "
        rf"parentheses. Each value is a mean over {N_SEEDS} seeds of "
        rf"{N_SPLITS}-fold cross-validation. Ranks 1--{TOP_K} are the subset the "
        rf"SHAP-IMV rows of Table~\ref{{{METRICS_LABEL}}} refit; anything beyond "
        rf"is the feature that method drops. {specification_note()}}}",
        r"\resizebox{\textwidth}{!}{",
        r"    \begin{tabular}{" + "c" * (2 + depth) + "}",
        r"        \toprule",
        r"         \multirow{2}{*}{Dataset} & \multirow{2}{*}{Model} & "
        + rf"\multicolumn{{{depth}}}{{c}}{{Rank by mean SHAP-IMV}}\\",
        r"         &  & " + " & ".join(str(rank) for rank in range(1, depth + 1))
        + r" \\",
        r"        \midrule",
    ]
    for position, result in enumerate(results.values()):
        if position:
            lines.append(r"        \midrule")
        lines.append(rf"        \multirow{{{len(MODEL_NAMES)}}}{{*}}{{{result['title']}}}")
        for offset, model in enumerate(MODEL_NAMES):
            ranked = result["ranking"].query("model == @model").sort_values("rank")
            cells = [ranking_cell(row) for _, row in ranked.iterrows()]
            cells += [""] * (depth - len(cells))   # a shorter feature set pads out
            lines.append("        & "
                         + " & ".join(shaded([MODEL_LABELS[model]] + cells, offset))
                         + r"\\")
    lines += [
        r"        \bottomrule",
        rf"        \label{{{RANKING_LABEL}}}",
        r"    \end{tabular}}",
        r"    \vspace{-0.4cm}",
        r"\end{table*}",
    ]
    return "\n".join(lines)

write_latex(ranking_latex(), TABLES / "shap_imv_ranking.tex", packages=PACKAGES)
print(ranking_latex())
display(combined_ranking.pivot_table(
    index=["dataset", "model"], columns="rank", values="feature",
    aggfunc="first").reindex(
        [(result["title"], model) for result in results.values()
         for model in MODEL_NAMES]))

## 4. Figures — the same ranking, with its spread across seeds

Two figures rather than one: Titanic on its own, as the published figure has it, and the other two examples panelled together. Every panel is lettered and names its own example and estimator — `a.` to `c.` for Titanic, `a.` to `f.` down the combined figure — with the letter set flush to the y-axis. Bars are in the ranking's order and the spread across seeds is the error bar.

Neither figure carries an overall title, so the protocol behind it — mean over ten seeds, error bars the spread across those seeds — belongs in the LaTeX caption that places it.

Every feature the example carries gets a bar; nothing is truncated here. What the bars cannot show is a variable the example never scored — the feature set is fixed in each example notebook's `FEATURES`, because exact Shapley costs one evaluation per subset of it.

Rows share a y-axis and columns do not: the three estimators of one example are meant to be read against each other, two examples are not. A negative bar would mean the feature *reduced* held-out information; none of these do.

In [ ]:
# One figure per group, so Titanic stands alone and the two remaining examples
# share a panelled figure. Adding an example is a name in this tuple.
FIGURE_GROUPS = (("titanic", ("shap_imv_titanic",)),
                 ("breast_cancer_adult_income",
                  ("shap_imv_breast_cancer", "shap_imv_adult_income")))
PANEL_LETTERS = "abcdefgh"

def ranking_figure(examples):
    """Mean SHAP-IMV per feature: one row per example, one column per estimator."""
    rows = [results[example] for example in examples]
    figure, axes = plt.subplots(
        len(rows), len(MODEL_NAMES),
        figsize=shared_style.figure_size(len(rows), len(MODEL_NAMES)),
        sharey="row", squeeze=False, layout="constrained")
    for row, result in enumerate(rows):
        for column, model in enumerate(MODEL_NAMES):
            axis = axes[row, column]
            # dropna only bites on the fallback ranking, where a feature below
            # the cut has a rank but no value left to draw.
            ranked = (result["ranking"].query("model == @model")
                      .sort_values("rank").dropna(subset=["mean"]))
            labels = [FEATURE_LABELS.get(name, name).replace(r"$\times$", "×")
                      for name in ranked["feature"]]
            axis.bar(labels, ranked["mean"], yerr=ranked["std"].fillna(0),
                     **shared_style.bar_style(len(ranked)))
            axis.axhline(0, color=shared_style.AXIS_COLOR,
                         linewidth=shared_style.EDGE_WIDTH)
            axis.set_xticks(range(len(labels)))
            axis.set_xticklabels(labels, rotation=45, ha="right")
            # Letters run across the row and then down, and each panel names
            # what it shows, so the figure needs no title of its own. loc="left"
            # sets the label flush to the y-axis rather than over the panel.
            letter = PANEL_LETTERS[row * len(MODEL_NAMES) + column]
            axis.set_title(f"{letter}. {result['title']} - {MODEL_TITLES[model]}",
                           loc="left", **shared_style.TITLE_KWARGS)
            if column == 0:
                axis.set_ylabel("Mean SHAP-IMV", **shared_style.LABEL_KWARGS)
    return figure

written = {}
for name, examples in FIGURE_GROUPS:
    figure = ranking_figure(examples)
    written[name] = save_figure(figure, FIGURES / f"shap_imv_results__{name}")["pdf"]
    plt.show()
{name: relative_path(path) for name, path in written.items()}